# Example-16: Joined frequency (method, with noise)

In [1]:
# Import

import numpy
import torch
import nufft
import yaml

import sys
sys.path.append('..')

from harmonica.util import LENGTH
from harmonica.statistics import weighted_mean, weighted_variance
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

True
16


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Load model data

with open('../config.yaml', 'r') as stream:
    config = yaml.safe_load(stream)
    config = {key: config[key] for key in sorted(config.keys(), key=lambda name: config[name]['TIME'])}

# Set normalized positions (used as locations with NUFFT)

position = numpy.array([value['TIME'] for key, value in config.items() if value['TYPE'] == 'MONITOR'])/LENGTH

# Set normalized accumulated phase advance between given monitor location and the next monitor location (used as locations with NUFFT)

phase = numpy.array([value['FX'] for key, value in config.items() if value['TYPE'] == 'MONITOR' or key == 'TAIL'])
*_, tune = phase
phase = numpy.diff(phase)
phase = numpy.cumsum(phase)/tune
start, *_ = phase
phase = phase - start
tune = tune/(2.0*numpy.pi)

# Set window

w = Window(4096, name='cosine_window', order=2.0, dtype=dtype, device=device)
print(w)

# Load TbT data from file and add noise

d = Data.from_file(54, w, '../virtual_tbt.npy')
s = 1.0E-4*torch.ones(54, dtype=dtype, device=device)
d.add_noise(s)
d.data.copy_(d.work)
print(d)

# Initialize Frequency instance

f = Frequency(d)
print(f)

# Compute reference frequency

d.window_remove_mean()
d.window_apply()
f('parabola')
d.reset()
mean = 9.0 - torch.mean(f.frequency).item()
std = torch.std(f.frequency).item()
print(tune)
print(mean)
print(abs(mean-tune))
print(std)

# Compute mixed frequencies in given range

length = 32
f1 = f.compute_joined_frequency(length=length, f_range=(8.5,9.0), order=1.0, normalize=True)
f2 = f.compute_joined_frequency(length=length, f_range=(8.5,9.0), order=1.0, normalize=True, position=position)
f3 = f.compute_joined_frequency(length=length, f_range=(8.5,9.0), order=1.0, normalize=True, position=phase)
print(f1.cpu().numpy())
print(f2.cpu().numpy())
print(f3.cpu().numpy())
print((f1 - tune).abs().cpu().numpy())
print((f2 - tune).abs().cpu().numpy())
print((f3 - tune).abs().cpu().numpy())

# Clean

del w
del d
del f
del f1, f2, f3
if device != torch.device('cpu'):
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

Window(4096, 'cosine_window', 2.0)
Data(54, Window(4096, 'cosine_window', 2.0))
Frequency(Data(54, Window(4096, 'cosine_window', 2.0)), f_range=(0.0, 0.5))
8.536883098737361
8.536883110185704
1.1448342718267668e-08
3.2638285414381684e-07
[8.53125    8.53685619 8.53684779]
[8.53676896 8.53685496 8.53685486]
[8.53676896 8.53684289 8.53684285]
[5.63309874e-03 2.69066077e-05 3.53039650e-05]
[1.14135217e-04 2.81343766e-05 2.82372960e-05]
[1.14135217e-04 4.02046699e-05 4.02476232e-05]
